In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import joblib
from torch.utils.data import DataLoader, TensorDataset

In [ ]:
torch.manual_seed(42)
np.random.seed(42)

In [ ]:
class TabularGenerator(nn.Module):
    def __init__(self, latent_dim, output_dim):
        super(TabularGenerator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(latent_dim, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh()                         # Tanh ensures values stay bounded within scaled limits
        )
    
    def forward(self, z):
        return self.model(z)

In [ ]:
class TabularDiscriminator(nn.Module):
    def __init__(self, input_dim):
        super(TabularDiscriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()                       # Outputs probability of data being real (1) or fake (0)
        )

    def forward(self, x):
        return self.model(x)

In [ ]:
def train_tabular_gan(X_minority, epochs=150, batch_size=64, latent_dim=32):
    input_dim = X_minority.shape[1]

    # Initialize networks
    generator = TabularGenerator(latent_dim, input_dim)
    discriminator = TabularDiscriminator(input_dim)

    # Loss and Optimizers
    criterion = nn.BCELoss()
    optimizer_G = optim.Adam(generator.parameters(), lr=0.0002, betas=(0.5, 0.999))
    optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002, betas=(0.5, 0.999))

    # Prepare DataLoader
    dataset = TensorDataset(torch.FloatTensor(X_minority))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

    for epoch in range(epochs):
        for i, (real_samples,) in enumerate(dataloader):
            current_batch_size = real_samples.size(0)

            # Labels for Binary Cross Entropy
            real_labels = torch.ones(current_batch_size, 1)
            fake_labels = torch.zeros(current_batch_size, 1)
            
            # ---------------------
            #  Train Discriminator
            # ---------------------
            optimizer_D.zero_grad()
            
            # Test real samples
            outputs_real = discriminator(real_samples)
            loss_real = criterion(outputs_real, real_labels)
            
            # Test fake samples generated from latent noise
            z = torch.randn(current_batch_size, latent_dim)
            fake_samples = generator(z)
            outputs_fake = discriminator(fake_samples.detach())
            loss_fake = criterion(outputs_fake, fake_labels)
            
            # Backpropagate Discriminator
            loss_D = loss_real + loss_fake
            loss_D.backward()
            optimizer_D.step()
            
            # -----------------
            #  Train Generator
            # -----------------
            optimizer_G.zero_grad()
            
            # We want the discriminator to think the fake samples are real
            outputs_g_fake = discriminator(fake_samples)
            loss_G = criterion(outputs_g_fake, real_labels)
            
            # Backpropagate Generator
            loss_G.backward()
            optimizer_G.step()
            
        if (epoch + 1) % 50 == 0 or epoch == 0:
            print(f"    [Epoch {epoch+1}/{epochs}] Loss D: {loss_D.item():.4f} | Loss G: {loss_G.item():.4f}")
            
    return generator


In [ ]:
if __name__ == "__main__":
    print("[*] Loading preprocessed data splits from Feature Engineering phase...")
    try:
        X_train = np.load('X_train_processed.npy')
        y_train = np.load('y_train.npy')
        target_encoder = joblib.load('attack_classes.pkl')
    except FileNotFoundError:
        print("[-] Missing input arrays. Run 'feature_engineer.py' first!")
        exit()

    # Identify Class Distribution
    classes, counts = np.unique(y_train, return_counts=True)
    max_samples = np.max(counts)
    normal_class_idx = classes[np.argmax(counts)]  # Assuming the majority class is 'Normal'
    
    print(f"[+] Majority baseline target size per class: {max_samples} rows.")
    
    X_augmented_list = [X_train]
    y_augmented_list = [y_train]
    
    # Loop through each minority class and generate synthetic records
    for class_idx in classes:
        if class_idx == normal_class_idx:
            continue
            
        class_name = target_encoder.inverse_transform([class_idx])[0]
        class_mask = (y_train == class_idx)
        X_minority = X_train[class_mask]
        samples_to_generate = max_samples - len(X_minority)
        
        if samples_to_generate <= 0:
            continue
            
        print(f"[*] Training Generative Adversarial Network for minority class: '{class_name}'...")
        print(f"    Target: Fabricating {samples_to_generate} synthetic network signatures...")
        
        # Train the PyTorch GAN
        gen_model = train_tabular_gan(X_minority, epochs=100, batch_size=32)
        
        # Generate synthetic vectors
        z_noise = torch.randn(samples_to_generate, 32)
        with torch.no_grad():
            synthetic_features = gen_model(z_noise).numpy()
            
        synthetic_labels = np.full(samples_to_generate, class_idx)
        
        X_augmented_list.append(synthetic_features)
        y_augmented_list.append(synthetic_labels)
        print(f"[+] Synthesis complete for class '{class_name}'.")

    # Save final balanced files
    X_train_balanced = np.concatenate(X_augmented_list, axis=0)
    y_train_balanced = np.concatenate(y_augmented_list, axis=0)
    
    print("\n" + "="*40)
    print("[+] GAN AUGMENTATION ENGINE COMPLETE")
    print(f"    Original Training Data Shape: {X_train.shape}")
    print(f"    Balanced Training Data Shape: {X_train_balanced.shape}")
    print("="*40)
    
    np.save('X_train_balanced.npy', X_train_balanced)
    np.save('y_train_balanced.npy', y_train_balanced)
    print("[*] Balanced data matrices exported as 'X_train_balanced.npy' and 'y_train_balanced.npy'. Ready for XGBoost Model.")